In [1]:
!pip install -q "Pillow<12.0.0" --force-reinstall
!pip install -q transformers accelerate bitsandbytes sentencepiece protobuf
!pip install -q langgraph langchain langchain-core langchain-community
!pip install -q pdfplumber pydantic reportlab

print("Installations completed!")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pdfplumber 0.11.10 requires Pillow>=12.2.0, but you have pillow 11.3.0 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.3.0 which is incompatible.
Installations completed!


In [2]:
import os
import json
import torch
from typing import List, Optional, Dict, Any, TypedDict
from pydantic import BaseModel, Field

from langgraph.graph import StateGraph, END
import pdfplumber

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

/usr/local/lib/python3.12/dist-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


CUDA available: True
GPU: Tesla T4


In [3]:
# Cell: Install Groq
!pip install -q langchain-groq

In [4]:
from langchain_groq import ChatGroq

GROQ_API_KEY = "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"

llm = ChatGroq(
    model = "openai/gpt-oss-120b",
    temperature = 0.2,
    api_key = GROQ_API_KEY
)

print("openai/gpt-oss-120b (via Groq) loaded successfully!")

openai/gpt-oss-120b (via Groq) loaded successfully!


In [5]:
response = llm.invoke("Say 'CareerForge AI with openai/gpt-oss-120b is ready' and nothing else.")
print(response.content)

CareerForge AI with openai/gpt-oss-120b is ready


In [6]:
def call_llm(prompt: str, system_prompt: str = None, max_tokens: int = 1500) -> str:
    messages = []
    if system_prompt:
        messages.append(("system",system_prompt))
    messages.append(("human",prompt))

    response = llm.invoke(messages)
    return response.content.strip()

print("New call_llm ready!")

New call_llm ready!


In [7]:
from typing import List, Optional, Dict
from pydantic import BaseModel, Field
from typing import TypedDict

class Education(BaseModel):
    degree: str = ""
    institution: str = ""
    years: str = ""
    details: Optional[str] = None

class Experience(BaseModel):
    title: str = ""
    company: str = ""
    duration: str = ""
    bullets: List[str] = Field(default_factory=list)

class Project(BaseModel):
    name: str = ""
    technologies: Optional[str] = None
    bullets: List[str] = Field(default_factory=list)

class Skills(BaseModel):
    technical: List[str] = Field(default_factory=list)
    tools: List[str] = Field(default_factory=list)
    soft: List[str] = Field(default_factory=list)
    raw_text: Optional[str] = None

class StructuredResume(BaseModel):
    name: str = ""
    email: Optional[str] = None
    phone: Optional[str] = None
    location: Optional[str] = None
    linkedin: Optional[str] = None
    headline: Optional[str] = None
    summary: Optional[str] = None
    education: List[Education] = Field(default_factory=list)
    experience: List[Experience] = Field(default_factory=list)
    projects: List[Project] = Field(default_factory=list)
    skills: Skills = Field(default_factory=Skills)
    achievements: List[str] = Field(default_factory=list)
    certifications: List[str] = Field(default_factory=list)
    raw_text: Optional[str] = None

class ResumeAnalysis(BaseModel):
    overall_score: float = 0.0
    strengths: List[str] = Field(default_factory=list)
    weaknesses: List[str] = Field(default_factory=list)
    missing_skills_for_role: List[str] = Field(default_factory=list)
    recommended_certifications: List[str] = Field(default_factory=list)
    recommended_projects: List[str] = Field(default_factory=list)
    improvement_areas: List[str] = Field(default_factory=list)
    feedback_summary: str = ""
    approval_status: str = "Needs Improvement"   # "Approved" or "Needs Improvement"

class ResumeSuggestion(BaseModel):
    original_bullet: str
    improved_bullet: str
    reason: str

class GeneratedContent(BaseModel):
    improved_summary: Optional[str] = None
    resume_suggestions: List[ResumeSuggestion] = Field(default_factory=list)
    interview_questions: List[str] = Field(default_factory=list)
    full_tailored_resume_text: Optional[str] = None

print("Pydantic models ready!")

Pydantic models ready!


In [8]:
import pdfplumber
import json

def extract_text_from_pdf(pdf_path: str) -> str:
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
    return text.strip()

def clean_llm_json(response: str) -> dict:
    response = response.strip()
    if "```json" in response:
        response = response.split("```json")[1].split("```")[0]
    elif "```" in response:
        response = response.split("```")[1].split("```")[0]
    response = response.strip()
    try:
        return json.loads(response)
    except:
        start = response.find("{")
        end = response.rfind("}") + 1
        if start != -1 and end > start:
            return json.loads(response[start:end])
        raise ValueError("Could not parse JSON from LLM response")

def safe_float(value, default=0.0):
    try:
        return float(value) if value is not None else default
    except:
        return default

def normalize_resume_data(data: dict) -> dict:
    if "projects" in data and isinstance(data["projects"], list):
        for project in data["projects"]:
            if isinstance(project.get("technologies"), list):
                project["technologies"] = " • ".join(project["technologies"])
            elif project.get("technologies") is None:
                project["technologies"] = ""
    
    if "achievements" in data and isinstance(data["achievements"], list):
        cleaned = []
        for item in data["achievements"]:
            if isinstance(item, dict):
                cleaned.append(item.get("text") or item.get("title") or str(item))
            else:
                cleaned.append(str(item))
        data["achievements"] = cleaned

    if "certifications" in data and isinstance(data["certifications"], list):
        cleaned = []
        for item in data["certifications"]:
            if isinstance(item, dict):
                cleaned.append(item.get("text") or item.get("name") or str(item))
            else:
                cleaned.append(str(item))
        data["certifications"] = cleaned

    if "skills" in data and isinstance(data["skills"], dict):
        for key in ["technical", "tools", "soft"]:
            if key in data["skills"] and isinstance(data["skills"][key], str):
                data["skills"][key] = [s.strip() for s in data["skills"][key].split(",")]
    
    return data

def parse_resume_with_llm(raw_text: str) -> StructuredResume:
    """Robust resume parser - this is the main function we use."""
    system_prompt = """You are an expert resume parser. 
Extract information and return ONLY a valid JSON object.
Important extraction rules:
- education: extract ALL degrees, institutions, years, CGPA/details if present
- certifications: extract ALL certificates as plain strings
- achievements: extract awards, hackathons, publications, honors as plain strings
- If a section exists in the resume, do NOT return empty lists
- Do not invent education/certs/achievements that are not in the text
No markdown, no explanation, just pure JSON."""

    user_prompt = f"""
Extract information from this resume and return JSON in this exact structure:

{{
  "name": "string",
  "email": "string or null",
  "phone": "string or null",
  "location": "string or null",
  "linkedin": "string or null",
  "headline": "string or null",
  "summary": "string or null",
  "education": [
    {{"degree": "string", "institution": "string", "years": "string", "details": "string or null"}}
  ],
  "experience": [
    {{
      "title": "string",
      "company": "string",
      "duration": "string",
      "bullets": ["string"]
    }}
  ],
  "projects": [
    {{
      "name": "string",
      "technologies": "string",
      "bullets": ["string"]
    }}
  ],
  "skills": {{
    "technical": ["string"],
    "tools": ["string"],
    "soft": ["string"],
    "raw_text": "string"
  }},
  "achievements": ["string"],
  "certifications": ["string"]
}}

Resume:
\"\"\"
{raw_text[:3800]}
\"\"\"
"""
    response = call_llm(user_prompt, system_prompt=system_prompt, max_tokens=2000)
    
    try:
        data = clean_llm_json(response)
        data = normalize_resume_data(data)
        data["raw_text"] = raw_text
        return StructuredResume(**data)
    except Exception as e:
        print("Parsing error:", str(e))
        print("LLM Response (first 600 chars):\n", response[:600])
        return StructuredResume(raw_text=raw_text)

print("Robust Resume Parser (parse_resume_with_llm) ready!")

Robust Resume Parser (parse_resume_with_llm) ready!


In [9]:
# ====================== IMPROVED RESUME ANALYZER ======================

from typing import List, Optional, Dict
from pydantic import BaseModel, Field

def analyze_resume(structured_resume: StructuredResume, target_role: str = None) -> ResumeAnalysis:
    """
    Improved Resume Analyzer with:
    - Anti-hallucination
    - Uses Projects, Achievements, Certifications, Education
    - Career Advisor suggestions
    - Realistic scoring
    """

    system_prompt = """You are a strict and honest technical career coach and resume reviewer.
Rules you must follow:
1. NEVER invent experience, companies, projects, or skills that are not in the resume.
2. Only use information that actually exists in the resume.
3. Be realistic with scoring (freshers should score lower).
4. If the resume is already excellent for the target role (score > 90), set approval_status to "Approved".
5. Return ONLY valid JSON."""

    # Experience
    exp_text = ""
    for exp in structured_resume.experience:
        exp_text += f"\n{exp.title} at {exp.company} ({exp.duration})\n"
        for b in exp.bullets:
            exp_text += f"  • {b}\n"

    # Projects
    projects_text = ""
    for p in structured_resume.projects:
        projects_text += f"- {p.name} ({p.technologies or ''}): {', '.join(p.bullets)}\n"

    # Education
    education_text = ""
    for edu in structured_resume.education:
        education_text += f"- {edu.degree} | {edu.institution} | {edu.years}"
        if edu.details:
            education_text += f" | {edu.details}"
        education_text += "\n"
    if not education_text:
        education_text = "None"

    achievements_text = "\n".join([f"- {a}" for a in structured_resume.achievements]) or "None"
    certifications_text = "\n".join([f"- {c}" for c in structured_resume.certifications]) or "None"
    skills_text = ", ".join(
        structured_resume.skills.technical +
        structured_resume.skills.tools +
        structured_resume.skills.soft
    )

    user_prompt = f"""
Analyze this resume honestly for the target role.

Target Role: {target_role or 'Not specified'}

=== CANDIDATE RESUME ===
Name: {structured_resume.name}
Headline: {structured_resume.headline or 'Not provided'}
Summary: {structured_resume.summary or 'Not provided'}
Skills: {skills_text or 'Not provided'}

Experience:
{exp_text or 'No experience listed'}

Projects:
{projects_text or 'No projects listed'}

Education:
{education_text}

Achievements:
{achievements_text}

Certifications:
{certifications_text}

=== INSTRUCTIONS ===
Return JSON in this exact format:

{{
  "overall_score": 75.0,
  "strengths": ["specific strength 1", "specific strength 2"],
  "weaknesses": ["specific weakness 1", "specific weakness 2"],
  "missing_skills_for_role": ["skill missing for the target role"],
  "recommended_certifications": ["valuable certification to pursue"],
  "recommended_projects": ["project idea that would attract recruiters"],
  "improvement_areas": ["actionable improvement"],
  "feedback_summary": "2-4 sentence honest overall assessment",
  "approval_status": "Approved" or "Needs Improvement"
}}

Scoring Guide:
- 90-100: Excellent for the target role -> approval_status = "Approved"
- 75-89: Strong but needs some improvements
- 60-74: Average, clear gaps exist
- Below 60: Weak / Fresher level

Important:
- Use achievements, certifications, education, and projects in your evaluation when present.
- Do not invent anything not in the resume.
"""

    response = call_llm(user_prompt, system_prompt=system_prompt, max_tokens=1800)

    try:
        data = clean_llm_json(response)

        score = safe_float(data.get("overall_score"))
        status = data.get("approval_status", "Needs Improvement")

        if score > 90:
            status = "Approved"

        return ResumeAnalysis(
            overall_score=score,
            strengths=[str(s) for s in data.get("strengths", [])],
            weaknesses=[str(w) for w in data.get("weaknesses", [])],
            missing_skills_for_role=[str(s) for s in data.get("missing_skills_for_role", [])],
            recommended_certifications=[str(c) for c in data.get("recommended_certifications", [])],
            recommended_projects=[str(p) for p in data.get("recommended_projects", [])],
            improvement_areas=[str(i) for i in data.get("improvement_areas", [])],
            feedback_summary=str(data.get("feedback_summary", "")),
            approval_status=status
        )
    except Exception as e:
        print("Analysis error:", str(e))
        print("Response:", response[:700])
        return ResumeAnalysis(feedback_summary=f"Analysis failed: {str(e)}")


print("Improved Resume Analyzer ready!")

Improved Resume Analyzer ready!


In [10]:
# ====================== FULL TEST FROM SCRATCH ======================

# 1. Path of the resume
resume_path = "/kaggle/input/datasets/shivammusk/candidates-resumes/resumes_v2/Resume_01_Aarav_Mehta.pdf"

# 2. Extract text from PDF
raw_text = extract_text_from_pdf(resume_path)
print("Text extracted. Length:", len(raw_text))

# 3. Parse the resume
structured = parse_resume_with_llm(raw_text)

print("\n===== PARSED RESUME =====")
print("Name     :", structured.name)
print("Headline :", structured.headline)
print("Email    :", structured.email)
print("Skills   :", structured.skills.technical[:6])
print("Experience entries:", len(structured.experience))
print("Projects :", len(structured.projects))
print("Achievements:", structured.achievements)
print("Certifications:", structured.certifications)

# 4. Analyze the resume
analysis = analyze_resume(structured, target_role="Backend Engineer")

print("\n===== RESUME ANALYSIS =====")
print(f"Score            : {analysis.overall_score}/100")
print(f"Approval Status  : {analysis.approval_status}")

print("\nStrengths:")
for s in analysis.strengths:
    print(f"  ✓ {s}")

print("\nWeaknesses:")
for w in analysis.weaknesses:
    print(f"  ✗ {w}")

print("\nMissing Skills for Role:")
for s in analysis.missing_skills_for_role:
    print(f"  → {s}")

print("\nRecommended Certifications:")
for c in analysis.recommended_certifications:
    print(f"  → {c}")

print("\nRecommended Projects:")
for p in analysis.recommended_projects:
    print(f"  → {p}")

print("\nFeedback:")
print(analysis.feedback_summary)

Text extracted. Length: 1617

===== PARSED RESUME =====
Name     : AARAV MEHTA
Headline : Backend Engineer | Distributed Systems | Python & Go
Email    : aarav.mehta.dev@gmail.com
Skills   : ['Python', 'Go', 'Java', 'SQL', 'FastAPI', 'Django']
Experience entries: 2
Projects : 1
Achievements: ['Winner – Smart India Hackathon 2022', 'Published paper on distributed caching at college symposium']
Certifications: ['AWS Certified Developer – Associate']

===== RESUME ANALYSIS =====
Score            : 82.0/100
Approval Status  : Needs Improvement

Strengths:
  ✓ 3 years of production experience building high‑throughput APIs and microservices
  ✓ Demonstrated performance impact (42% latency reduction, 99.95% reliability) and successful monolith‑to‑microservices migration
  ✓ Strong core backend stack (Python, Go, FastAPI, Django, gRPC, PostgreSQL, Redis, Kafka, Docker, Kubernetes, AWS)

Weaknesses:
  ✗ Limited exposure to observability tooling (e.g., Prometheus, Grafana, OpenTelemetry)
  ✗ No 

In [11]:
structured = parse_resume_with_llm(raw_text)

print("Name:", structured.name)
print("Education:", structured.education)
print("Achievements:", structured.achievements)
print("Certifications:", structured.certifications)

Name: AARAV MEHTA
Education: [Education(degree='B.Tech Computer Science', institution='IIT Bombay', years='2019 – 2023', details='CGPA: 8.9/10')]
Achievements: ['Winner – Smart India Hackathon 2022', 'Published paper on distributed caching at college symposium']
Certifications: ['AWS Certified Developer – Associate']


In [12]:
def generate_content_from_resume(
    structured_resume: StructuredResume,
    analysis: ResumeAnalysis,
    target_role: str = None
) -> GeneratedContent:
    """
    Generates:
    - Improved Summary
    - Resume Suggestions
    - Interview Questions
    - Full Improved Resume + Career Growth Recommendations
    """

    system_prompt = """You are an expert career coach and resume writer.
Return ONLY valid JSON. Never invent work experience that does not exist in the original resume."""

    is_approved = analysis.approval_status == "Approved" or analysis.overall_score > 90

    strengths_text = "\n".join([f"- {s}" for s in analysis.strengths[:3]])
    weaknesses_text = "\n".join([f"- {w}" for w in analysis.weaknesses[:3]])
    missing_skills = ", ".join(analysis.missing_skills_for_role) or "None"
    recommended_certs = ", ".join(analysis.recommended_certifications) or "None"
    recommended_projects = "\n".join([f"- {p}" for p in analysis.recommended_projects]) or "None"

    exp_text = ""
    for exp in structured_resume.experience[:2]:
        exp_text += f"{exp.title} at {exp.company}: " + " | ".join(exp.bullets[:2]) + "\n"

    projects_text = ""
    for p in structured_resume.projects[:2]:
        projects_text += f"- {p.name}: {', '.join(p.bullets[:1])}\n"

    education_text = ""
    for edu in structured_resume.education:
        education_text += f"- {edu.degree}, {edu.institution} ({edu.years})"
        if edu.details:
            education_text += f" | {edu.details}"
        education_text += "\n"
    if not education_text:
        education_text = "Not provided"

    # -------- Call 1: Summary + Suggestions + Questions --------
    if is_approved:
        prompt1 = f"""
The candidate's resume is already strong (score {analysis.overall_score}).
Rules:
- If education data is provided above, include it exactly under EDUCATION
- Do NOT write "Details available upon request" when education is present
- Always include Achievements and Certifications when provided

Name: {structured_resume.name}
Target Role: {target_role or 'Software Engineer'}
Current Summary: {structured_resume.summary}

Return JSON:
{{
  "improved_summary": "Slightly polished professional summary",
  "resume_suggestions": [],
  "interview_questions": ["Q1", "Q2", "Q3", "Q4", "Q5", "Q6", "Q7", "Q8"]
}}
"""
    else:
        prompt1 = f"""
Generate improved content for this candidate.

Name: {structured_resume.name}
Target Role: {target_role or 'Software Engineer'}
Current Summary: {structured_resume.summary}

Strengths:
{strengths_text}

Weaknesses:
{weaknesses_text}

Missing Skills: {missing_skills}

Experience:
{exp_text}

Projects:
{projects_text}

Education:
{education_text}

Return JSON:
{{
  "improved_summary": "Strong 3-line professional summary tailored to the target role",
  "resume_suggestions": [
    {{"original_bullet": "original text", "improved_bullet": "stronger version", "reason": "why better"}},
    {{"original_bullet": "original text", "improved_bullet": "stronger version", "reason": "why better"}},
    {{"original_bullet": "original text", "improved_bullet": "stronger version", "reason": "why better"}}
  ],
  "interview_questions": ["Q1", "Q2", "Q3", "Q4", "Q5", "Q6", "Q7", "Q8"]
}}
"""

    response1 = call_llm(prompt1, system_prompt, max_tokens=1600)

    improved_summary = None
    suggestions = []
    questions = []

    try:
        data1 = clean_llm_json(response1)
        improved_summary = data1.get("improved_summary")
        for s in data1.get("resume_suggestions", []):
            suggestions.append(ResumeSuggestion(
                original_bullet=str(s.get("original_bullet", "")),
                improved_bullet=str(s.get("improved_bullet", "")),
                reason=str(s.get("reason", ""))
            ))
        questions = [str(q) for q in data1.get("interview_questions", [])]
    except Exception as e:
        print("Call 1 error:", e)

    # -------- Call 2: Full Improved Resume + Career Growth Section --------
    prompt2 = f"""
Create a complete improved resume in clean plain text, followed by a Career Growth section.

Name: {structured_resume.name}
Headline: {structured_resume.headline or target_role or 'Software Engineer'}
Improved Summary: {improved_summary or structured_resume.summary}
Skills: {', '.join(structured_resume.skills.technical[:10])}
Achievements: {', '.join(structured_resume.achievements)}
Certifications: {', '.join(structured_resume.certifications)}

Experience:
{exp_text}

Projects:
{projects_text}

Missing Skills to address: {missing_skills}
Recommended Certifications: {recommended_certs}
Recommended Projects:
{recommended_projects}

Write the output in this structure (plain text, NOT JSON body):

Name
Headline

SUMMARY
...

SKILLS
...

EXPERIENCE
...

PROJECTS
...

ACHIEVEMENTS
...

CERTIFICATIONS
...

EDUCATION
...

----------------------------------------
CAREER GROWTH RECOMMENDATIONS
----------------------------------------

Missing Skills to Learn:
- skill 1 (why it matters for the target role)
- skill 2

Recommended Projects to Build:
- project idea 1 (what skills it demonstrates)
- project idea 2

Recommended Certifications:
- Certification Name
  Where to learn: (e.g. Official docs / Coursera / freeCodeCamp / YouTube)

Next Steps:
- Short actionable advice on how to update the resume after learning these

Important:
- Do NOT invent fake work experience.
- Only recommend skills/projects/certs based on the gaps provided.

Return JSON:
{{
  "full_improved_resume": "the complete plain text content here including Career Growth section"
}}
"""

    response2 = call_llm(prompt2, system_prompt, max_tokens=1800)

    full_resume = None
    try:
        data2 = clean_llm_json(response2)
        full_resume = data2.get("full_improved_resume")
    except Exception as e:
        print("Call 2 error:", e)

    return GeneratedContent(
        improved_summary=improved_summary,
        resume_suggestions=suggestions,
        interview_questions=questions,
        full_tailored_resume_text=full_resume
    )

print("Updated Content Generator with Career Growth Recommendations ready!")

Updated Content Generator with Career Growth Recommendations ready!


In [13]:
generated = generate_content_from_resume(structured, analysis, target_role="Backend Engineer")

print("===== IMPROVED SUMMARY =====")
print(generated.improved_summary)

print("\n===== RESUME SUGGESTIONS =====")
for i, s in enumerate(generated.resume_suggestions, 1):
    print(f"\n{i}. Original : {s.original_bullet}")
    print(f"   Improved : {s.improved_bullet}")
    print(f"   Reason   : {s.reason}")

print("\n===== FULL IMPROVED RESUME =====")
print(generated.full_tailored_resume_text)

print("\n===== INTERVIEW QUESTIONS =====")
for i, q in enumerate(generated.interview_questions, 1):
    print(f"{i}. {q}")

===== IMPROVED SUMMARY =====
Backend Engineer with 3 years of experience designing high‑throughput, low‑latency APIs and microservices on Python and Go. Expert at optimizing performance (42% latency reduction) and driving reliability (99.95% uptime) at scale, with hands‑on experience in Docker, Kubernetes, AWS, PostgreSQL, Redis, and Kafka.

===== RESUME SUGGESTIONS =====

1. Original : Designed and scaled payment reconciliation service handling 50k+ transactions/day with 99.95% reliability
   Improved : Architected and scaled a payment reconciliation service processing >50k transactions daily, achieving 99.95% reliability and supporting seamless peak‑load spikes.
   Reason   : Adds verbs that highlight ownership (architected), quantifies load more clearly, and emphasizes reliability impact.

2. Original : Reduced p99 API latency by 42% through query optimization and caching strategies
   Improved : Reduced 99th‑percentile API latency by 42% by refactoring SQL queries, introducing Redi

In [14]:
# ====================== FINAL NODES + LANGGRAPH ======================

from langgraph.graph import StateGraph, END
from typing import TypedDict, Optional

class AgentState(TypedDict):
    resume_raw_text: str
    target_role: Optional[str]
    structured_resume: Optional[StructuredResume]
    resume_analysis: Optional[ResumeAnalysis]
    generated_content: Optional[GeneratedContent]
    error: Optional[str]

def resume_parser_node(state: dict) -> dict:
    print("→ Running Resume Parser...")
    raw_text = state.get("resume_raw_text", "")
    if not raw_text:
        state["error"] = "No resume text provided"
        return state
    try:
        structured = parse_resume_with_llm(raw_text)
        state["structured_resume"] = structured
        print(f"✓ Resume parsed: {structured.name}")
    except Exception as e:
        state["error"] = f"Parsing failed: {str(e)}"
        print("✗", state["error"])
    return state

def resume_analyzer_node(state: dict) -> dict:
    print("→ Running Resume Analyzer...")
    resume = state.get("structured_resume")
    target_role = state.get("target_role")
    if not resume:
        state["error"] = "No structured resume"
        return state
    try:
        analysis = analyze_resume(resume, target_role)
        state["resume_analysis"] = analysis
        print(f"✓ Analyzed | Score: {analysis.overall_score} | Status: {analysis.approval_status}")
    except Exception as e:
        state["error"] = f"Analysis failed: {str(e)}"
        print("✗", state["error"])
    return state

def generator_node(state: dict) -> dict:
    print("→ Running Content Generator...")
    resume = state.get("structured_resume")
    analysis = state.get("resume_analysis")
    target_role = state.get("target_role")
    if not resume or not analysis:
        state["error"] = "Missing resume or analysis"
        return state
    try:
        generated = generate_content_from_resume(resume, analysis, target_role)
        state["generated_content"] = generated
        print("✓ Content generated!")
        print(f"  - Summary      : {'Yes' if generated.improved_summary else 'No'}")
        print(f"  - Suggestions  : {len(generated.resume_suggestions)}")
        print(f"  - Questions    : {len(generated.interview_questions)}")
        print(f"  - Full Resume  : {'Yes' if generated.full_tailored_resume_text else 'No'}")
    except Exception as e:
        state["error"] = str(e)
        print("✗", state["error"])
    return state

def create_careerforge_graph():
    workflow = StateGraph(AgentState)
    workflow.add_node("parse_resume", resume_parser_node)
    workflow.add_node("analyze_resume", resume_analyzer_node)
    workflow.add_node("generate_content", generator_node)

    workflow.set_entry_point("parse_resume")
    workflow.add_edge("parse_resume", "analyze_resume")
    workflow.add_edge("analyze_resume", "generate_content")
    workflow.add_edge("generate_content", END)
    return workflow.compile()

careerforge_agent = create_careerforge_graph()
print("CareerForge AI Graph compiled!")

def run_careerforge(resume_pdf_path: str, target_role: str = None):
    print("=" * 65)
    print("         CareerForge AI - Phase 1 (Candidate Side)")
    print("=" * 65)

    resume_text = extract_text_from_pdf(resume_pdf_path)
    initial_state = {
        "resume_raw_text": resume_text,
        "target_role": target_role,
        "structured_resume": None,
        "resume_analysis": None,
        "generated_content": None,
        "error": None
    }

    final_state = careerforge_agent.invoke(initial_state)

    if final_state.get("error"):
        print("\nPipeline Error:", final_state["error"])
    else:
        print("\nPipeline Completed Successfully!")
    return final_state

print("Runner ready!")

CareerForge AI Graph compiled!
Runner ready!


In [15]:
# ====================== PHASE 2: RECRUITER SIDE ======================

In [16]:
# ====================== USE EXISTING SQLITE DATABASE ======================

import sqlite3
import json
from typing import List

# Change this path to where your DB file is
DB_PATH = "/kaggle/input/datasets/shivammusk/carreerforge-candidate-db/careerforge_candidates.db"

def get_connection():
    return sqlite3.connect(DB_PATH)

def get_candidate_count() -> int:
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT COUNT(*) FROM candidates")
    count = cursor.fetchone()[0]
    conn.close()
    return count

def load_all_candidates() -> List[StructuredResume]:
    """Load all candidates from the existing database."""
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM candidates")
    rows = cursor.fetchall()
    conn.close()

    candidates = []
    for row in rows:
        try:
            skills_data = json.loads(row[7]) if row[7] else {}
            experience_data = json.loads(row[8]) if row[8] else []
            projects_data = json.loads(row[9]) if row[9] else []
            achievements_data = json.loads(row[10]) if row[10] else []
            certifications_data = json.loads(row[11]) if row[11] else []
            education_data = json.loads(row[12]) if row[12] else []

            candidate = StructuredResume(
                name=row[1] or "",
                email=row[2],
                phone=row[3],
                location=row[4],
                headline=row[5],
                summary=row[6],
                skills=Skills(**skills_data) if skills_data else Skills(),
                experience=[Experience(**exp) for exp in experience_data],
                projects=[Project(**proj) for proj in projects_data],
                achievements=achievements_data,
                certifications=certifications_data,
                education=[Education(**edu) for edu in education_data],
                raw_text=row[13]
            )
            candidates.append(candidate)
        except Exception as e:
            print(f"Error loading candidate ID {row[0]}: {e}")

    return candidates

# Quick check
print("DB Path:", DB_PATH)
print("Total candidates in DB:", get_candidate_count())

candidates = load_all_candidates()
print("Loaded into memory:", len(candidates))
print("Sample:", [c.name for c in candidates[:5]])

DB Path: /kaggle/input/datasets/shivammusk/carreerforge-candidate-db/careerforge_candidates.db
Total candidates in DB: 27
Loaded into memory: 27
Sample: ['AARAV MEHTA', 'JORDAN LEE', 'PRIYA NAIR', 'SOFIA RAMIREZ', 'Kenji Tanaka']


In [17]:
candidates = load_all_candidates()
print(f"Total candidates loaded: {len(candidates)}")
print("\nSample candidates:")
for c in candidates[:5]:
    print(f"  - {c.name} | {c.headline} | Skills: {len(c.skills.technical)}")

Total candidates loaded: 27

Sample candidates:
  - AARAV MEHTA | Backend Engineer | Distributed Systems | Python & Go | Skills: 13
  - JORDAN LEE | Software Developer | Backend | Java & Spring | Skills: 6
  - PRIYA NAIR | Computer Science Graduate | Aspiring Backend Developer | Skills: 7
  - SOFIA RAMIREZ | Frontend Engineer | React TypeScript Design Systems | Skills: 12
  - Kenji Tanaka | Frontend Developer | React & JavaScript | Skills: 7


In [18]:
# ====================== PHASE 2: JD Analyzer ======================

class JobRequirements(BaseModel):
    job_title: str = ""
    company: Optional[str] = None
    location: Optional[str] = None
    required_skills: List[str] = Field(default_factory=list)
    preferred_skills: List[str] = Field(default_factory=list)
    experience_level: Optional[str] = None
    responsibilities: List[str] = Field(default_factory=list)
    keywords: List[str] = Field(default_factory=list)
    education_requirements: Optional[str] = None
    raw_text: Optional[str] = None

def parse_job_description(jd_text: str) -> JobRequirements:
    """
    Analyze a Job Description and extract structured requirements.
    """
    system_prompt = """You are an expert technical recruiter and job description analyzer.
Extract key information accurately. Return ONLY valid JSON. No markdown."""

    user_prompt = f"""
Analyze the following Job Description and return JSON in this exact structure:

{{
  "job_title": "string",
  "company": "string or null",
  "location": "string or null",
  "required_skills": ["skill1", "skill2"],
  "preferred_skills": ["skill1", "skill2"],
  "experience_level": "e.g. 2-4 years / Senior / Entry level",
  "responsibilities": ["responsibility1", "responsibility2"],
  "keywords": ["important keyword1", "keyword2"],
  "education_requirements": "string or null"
}}

Rules:
- required_skills = must-have skills
- preferred_skills = nice-to-have skills
- keywords = important technologies, domains, and soft skills
- Keep lists concise but complete

Job Description:
\"\"\"
{jd_text[:3500]}
\"\"\"
"""

    response = call_llm(user_prompt, system_prompt=system_prompt, max_tokens=1200)

    try:
        data = clean_llm_json(response)
        data["raw_text"] = jd_text
        return JobRequirements(**data)
    except Exception as e:
        print("JD Parsing error:", str(e))
        print("Response:", response[:600])
        return JobRequirements(raw_text=jd_text)

print("JD Analyzer ready!")

JD Analyzer ready!


In [19]:
sample_jd = """
We are looking for a Backend Engineer with 2+ years of experience.

Requirements:
- Strong proficiency in Python and Go
- Experience with microservices and distributed systems
- Knowledge of PostgreSQL, Redis, and Kafka
- Familiarity with Docker and Kubernetes
- Good understanding of system design

Preferred:
- Experience with AWS
- Prior work in fintech

Responsibilities:
- Design and build scalable backend services
- Optimize API performance
- Collaborate with cross-functional teams
"""

jd_result = parse_job_description(sample_jd)

print("===== JD ANALYSIS =====")
print("Job Title         :", jd_result.job_title)
print("Required Skills   :", jd_result.required_skills)
print("Preferred Skills  :", jd_result.preferred_skills)
print("Experience Level  :", jd_result.experience_level)
print("Keywords          :", jd_result.keywords)
print("Responsibilities  :", jd_result.responsibilities)

===== JD ANALYSIS =====
Job Title         : Backend Engineer
Required Skills   : ['Python', 'Go', 'Microservices', 'Distributed systems', 'PostgreSQL', 'Redis', 'Kafka', 'Docker', 'Kubernetes', 'System design']
Preferred Skills  : ['AWS', 'Fintech']
Experience Level  : 2+ years
Keywords          : ['Python', 'Go', 'Microservices', 'Distributed systems', 'PostgreSQL', 'Redis', 'Kafka', 'Docker', 'Kubernetes', 'System design', 'API performance', 'Cross-functional collaboration', 'AWS', 'Fintech']
Responsibilities  : ['Design and build scalable backend services', 'Optimize API performance', 'Collaborate with cross-functional teams']


In [20]:
# ====================== Matching Engine ======================

class CandidateMatch(BaseModel):
    candidate_name: str
    overall_score: float = 0.0
    llm_score: float = 0.0
    strengths: List[str] = Field(default_factory=list)
    gaps: List[str] = Field(default_factory=list)
    summary: str = ""
    interview_questions: List[str] = Field(default_factory=list)


def match_candidate_to_jd(candidate: StructuredResume, jd: JobRequirements) -> CandidateMatch:
    """
    LLM-only matching (no separate skills score).
    """
    system_prompt = """You are an expert technical recruiter.
Compare the candidate to the job description honestly and realistically.
Return ONLY valid JSON."""

    candidate_skills = ", ".join(candidate.skills.technical[:12])
    exp_summary = ""
    for exp in candidate.experience[:2]:
        exp_summary += f"{exp.title} at {exp.company}; "

    projects_summary = ", ".join([p.name for p in candidate.projects[:3]]) or "None"
    achievements_summary = ", ".join(candidate.achievements[:2]) or "None"
    certifications_summary = ", ".join(candidate.certifications[:2]) or "None"

    user_prompt = f"""
Job Title: {jd.job_title}
Required Skills: {', '.join(jd.required_skills)}
Preferred Skills: {', '.join(jd.preferred_skills)}
Experience Level: {jd.experience_level}
Responsibilities: {', '.join(jd.responsibilities[:4])}

Candidate: {candidate.name}
Headline: {candidate.headline}
Skills: {candidate_skills}
Experience: {exp_summary}
Projects: {projects_summary}
Achievements: {achievements_summary}
Certifications: {certifications_summary}

Return JSON:
{{
  "llm_score": 75.0,
  "strengths": ["strength 1", "strength 2"],
  "gaps": ["gap 1", "gap 2"],
  "summary": "2-sentence recruiter summary of fit"
}}

Scoring guide:
- 90-100: Excellent match
- 75-89: Strong match
- 60-74: Moderate match with clear gaps
- 40-59: Weak match
- Below 40: Poor match
"""

    response = call_llm(user_prompt, system_prompt, max_tokens=900)

    llm_score = 0.0
    strengths = []
    gaps = []
    summary = ""

    try:
        data = clean_llm_json(response)
        llm_score = safe_float(data.get("llm_score"))
        strengths = [str(s) for s in data.get("strengths", [])]
        gaps = [str(g) for g in data.get("gaps", [])]
        summary = str(data.get("summary", ""))
    except Exception as e:
        print(f"LLM match error for {candidate.name}:", e)

    return CandidateMatch(
        candidate_name = candidate.name,
        overall_score = llm_score,      # Only LLM score
        llm_score = llm_score,
        strengths = strengths,
        gaps = gaps,
        summary = summary
    )

print("Updated Matching Engine (LLM-only scoring) ready!")

Updated Matching Engine (LLM-only scoring) ready!


In [21]:
# Test matching on first candidate
test_candidate = candidates[0]
match = match_candidate_to_jd(test_candidate, jd_result)

print("===== MATCH TEST =====")
print(f"Candidate     : {match.candidate_name}")
print(f"Overall Score : {match.overall_score}")
print(f"LLM Score     : {match.llm_score}")
print(f"Strengths     : {match.strengths}")
print(f"Gaps          : {match.gaps}")
print(f"Summary       : {match.summary}")

===== MATCH TEST =====
Candidate     : AARAV MEHTA
Overall Score : 82.0
LLM Score     : 82.0
Strengths     : ['Strong proficiency in Python and Go', 'Hands‑on experience with microservices, system design, PostgreSQL, Redis, and Kafka', 'Demonstrated distributed systems knowledge (rate limiter project, research paper)', 'Fintech background through Razorpay role', 'AWS Certified Developer – Associate']
Gaps          : ['No explicit experience with Docker and Kubernetes mentioned', 'Limited evidence of production‑grade Go projects beyond Razorpay', 'Fintech preference satisfied, but AWS experience not shown beyond certification']
Summary       : Aarav is a strong match for the backend engineer role, bringing solid Python/Go and microservices expertise along with fintech experience. The main gaps are lack of stated Docker/Kubernetes hands‑on work, which could be addressed with targeted upskilling.


In [22]:
def rank_candidates(candidates: List[StructuredResume], jd: JobRequirements, top_n: int = 5):
    """
    Match all candidates against the JD and return Top N ranked by overall_score.
    """
    print(f"Matching {len(candidates)} candidates against: {jd.job_title}")
    print("-" * 50)

    matches = []

    for i, candidate in enumerate(candidates, 1):
        try:
            print(f"[{i}/{len(candidates)}] Matching: {candidate.name}")
            match = match_candidate_to_jd(candidate, jd)
            matches.append(match)
            print(f"    Score: {match.overall_score}")
        except Exception as e:
            print(f"    ✗ Error: {e}")

    matches.sort(key = lambda x: x.overall_score, reverse = True)

    top_matches = matches[:top_n]

    print("\n" + "="*50)
    print(f"Top {top_n} Candidates")
    for i, m in enumerate(top_matches, 1):
        print(f"{i}. {m.candidate_name} - {m.overall_score}/100")

    return top_matches

print("Ranking function ready!")

Ranking function ready!


In [23]:
# ====================== Generate Interview Questions for Top Candidates ======================

def generate_interview_questions_for_candidate(
    candidate: StructuredResume, 
    jd: JobRequirements, 
    match: CandidateMatch
) -> List[str]:
    """
    Generate 6-8 tailored interview questions based on JD + candidate profile.
    """
    system_prompt = "You are an expert technical interviewer. Return ONLY valid JSON."

    user_prompt = f"""
Generate interview questions for this candidate for the given role.

Job Title: {jd.job_title}
Required Skills: {', '.join(jd.required_skills[:8])}

Candidate: {candidate.name}
Headline: {candidate.headline}
Key Strengths: {', '.join(match.strengths[:3])}
Key Gaps: {', '.join(match.gaps[:2])}

Return JSON:
{{
  "interview_questions": [
    "Question 1",
    "Question 2",
    "Question 3",
    "Question 4",
    "Question 5",
    "Question 6",
    "Question 7",
    "Question 8"
  ]
}}
"""
    response = call_llm(user_prompt, system_prompt, max_tokens=900)
    
    try:
        data = clean_llm_json(response)
        return [str(q) for q in data.get("interview_questions", [])]
    except Exception as e:
        print(f"Question generation error for {candidate.name}:", e)
        return []

def enrich_top_candidates_with_questions(
    top_matches: List[CandidateMatch], 
    candidates: List[StructuredResume], 
    jd: JobRequirements
) -> List[CandidateMatch]:
    """
    Add interview questions to each top candidate.
    """
    name_to_candidate = {c.name.upper(): c for c in candidates}
    
    print("Generating interview questions for Top candidates...\n")
    
    for i, match in enumerate(top_matches, 1):
        candidate = name_to_candidate.get(match.candidate_name.upper())
        if not candidate:
            print(f"[{i}] Could not find full profile for {match.candidate_name}")
            continue
            
        print(f"[{i}] Generating questions for {match.candidate_name}")
        questions = generate_interview_questions_for_candidate(candidate, jd, match)
        match.interview_questions = questions
        print(f"    ✓ {len(questions)} questions generated")
    
    return top_matches

print("Interview Question Generator ready!")

Interview Question Generator ready!


In [24]:
# ====================== FINAL RECRUITER REPORT ======================

def print_recruiter_report(jd: JobRequirements, top_matches: List[CandidateMatch]):
    print("\n" + "="*70)
    print("           CAREERFORGE AI - RECRUITER SHORTLIST REPORT")
    print("="*70)
    
    print(f"\nJob Title          : {jd.job_title}")
    print(f"Required Skills    : {', '.join(jd.required_skills[:8])}")
    print(f"Experience Level   : {jd.experience_level}")
    print(f"Total Shortlisted  : {len(top_matches)}")
    
    for rank, match in enumerate(top_matches, 1):
        print("\n" + "-"*70)
        print(f"#{rank}  {match.candidate_name}  |  Match Score: {match.overall_score}/100")
        print("-"*70)
        
        print(f" LLM Score : {match.llm_score}")
        
        print("\nStrengths:")
        for s in match.strengths:
            print(f"  ✓ {s}")
        
        print("\nGaps:")
        for g in match.gaps:
            print(f"  ✗ {g}")
        
        print(f"\nSummary: {match.summary}")
        
        print("\nRecommended Interview Questions:")
        for i, q in enumerate(match.interview_questions[:6], 1):
            print(f"  {i}. {q}")
    
    print("\n" + "="*70)
    print("End of Report")
    print("="*70)




In [25]:
def run_recruiter_pipeline(jd_text: str, top_n: int = 5):
    """
    Full Recruiter-side pipeline:
    1. Analyze JD
    2. Load candidates from SQLite
    3. Hybrid match + rank
    4. Generate interview questions for Top N
    5. Print Recruiter Report
    """

    print("=" * 70)
    print("CareerForgeAI - Recruiter Pipeline")
    print("=" * 70)

    # Step 1: Analyze JD
    print("\n-> Step 1: Analyzing Job Description")
    jd = parse_job_description(jd_text)
    print(f"✓ Job Title: {jd.job_title}")
    print(f" Required Skills: {','.join(jd.required_skills[:6])}")

    # Step 2: Load candidates from SQLite
    print("\n-> Step 2: Loading candidates from database...")
    all_candidates = load_all_candidates()
    print(f"✓ Loaded {len(all_candidates)} candidates")

    if not all_candidates:
        print("✗ No candidates found in database")
        return None

    # Step 3: Match + Rank
    print("\n->Step 3: Matching and Ranking candidates...")
    top_matches = rank_candidates(all_candidates, jd, top_n=top_n)

    # Step 4: Generate Interview Questions
    print("\n→ Step 4: Generating interview questions...")
    top_matches = enrich_top_candidates_with_questions(top_matches, all_candidates, jd)

    # Step 5: Print Final Report
    print("\n→ Step 5: Generating Recruiter Report...")
    print_recruiter_report(jd, top_matches)

    return {
        "job_requirements": jd,
        "top_candidates": top_matches
    }

print("Recruiter Pipeline ready!")

Recruiter Pipeline ready!


In [26]:
!pip install -q chromadb sentence-transformers

In [27]:
# 1) Fix OpenTelemetry versions
!pip install -q -U "opentelemetry-api>=1.27.0" "opentelemetry-sdk>=1.27.0" "opentelemetry-exporter-otlp-proto-grpc>=1.27.0" "opentelemetry-instrumentation>=0.48b0"

# 2) Reinstall chromadb cleanly (optional but safer)
!pip install -q -U chromadb

print("Dependencies updated. Now RESTART the kernel, then re-run imports.")

Dependencies updated. Now RESTART the kernel, then re-run imports.


In [28]:
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer
import json

embedder = SentenceTransformer("all-MiniLM-L6-v2")

chroma_client = chromadb.PersistentClient(path="/kaggle/working/careerforge_chroma")

collection = chroma_client.get_or_create_collection(
    name = "candidates",
    metadata = {"hnsw:space":"cosine"}
)

print("Chroma collection ready. Current count:", collection.count())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Chroma collection ready. Current count: 0


In [29]:
def candidate_to_text(c: StructuredResume) -> str:
    """Create a searchable text representation of a candidate."""
    skills = ", ".join(c.skills.technical + c.skills.tools)
    exp_parts = []
    for e in c.experience[:3]:
        exp_parts.append(f"{e.title} at {e.company}: " + "; ".join(e.bullets[:2]))
    projects = "; ".join([f"{p.name} ({p.technologies or ''})" for p in c.projects[:3]])
    achievements = "; ".join(c.achievements[:3])
    certs = "; ".join(c.certifications[:3])

    return f"""
Name: {c.name}
Headline: {c.headline or ''}
Summary: {c.summary or ''}
Skills: {skills}
Experience: {' | '.join(exp_parts)}
Projects: {projects}
Achievements: {achievements}
Certifications: {certs}
""".strip()


def build_vector_index(candidates: list, reset: bool = False):
    """Embed all candidates and store in Chroma."""
    global collection

    if reset and collection.count() > 0:
        chroma_client.delete_collection("candidates")
        collection = chroma_client.get_or_create_collection(
            name="candidates",
            metadata={"hnsw:space": "cosine"}
        )

    if collection.count() > 0 and not reset:
        print(f"Index already has {collection.count()} items. Skipping rebuild.")
        print("Set reset=True if you want to rebuild.")
        return

    documents = []
    ids = []
    metadatas = []

    for i, c in enumerate(candidates):
        text = candidate_to_text(c)
        documents.append(text)
        ids.append(f"cand_{i}_{c.name.replace(' ', '_')}")
        metadatas.append({
            "name": c.name or "",
            "headline": c.headline or "",
            "email": c.email or ""
        })

    embeddings = embedder.encode(documents, show_progress_bar=True).tolist()

    # If reset left an empty collection, just add
    # If not reset but empty, also add
    if collection.count() > 0:
        # Clear by deleting and recreating for a clean rebuild
        chroma_client.delete_collection("candidates")
        collection = chroma_client.get_or_create_collection(
            name="candidates",
            metadata={"hnsw:space": "cosine"}
        )

    collection.add(
        documents=documents,
        embeddings=embeddings,
        metadatas=metadatas,
        ids=ids
    )
    print(f"✅ Indexed {len(documents)} candidates into Chroma.")

# Build index from SQLite-loaded candidates
# candidates = load_all_candidates()
# build_vector_index(candidates, reset=False)

In [30]:
candidates = load_all_candidates()
build_vector_index(candidates, reset=True)
print("Count:", collection.count())

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Indexed 27 candidates into Chroma.
Count: 27


In [31]:
def retrieve_candidates(jd_text: str, top_k: int = 8):
    jd_embedding = embedder.encode([jd_text]).tolist()

    results = collection.query(
        query_embeddings=jd_embedding,
        n_results=min(top_k, max(collection.count(), 1)),
        include=["documents", "metadatas", "distances"]
    )

    retrieved = []
    for i in range(len(results["ids"][0])):
        retrieved.append({
            "id": results["ids"][0][i],
            "name": results["metadatas"][0][i].get("name", ""),
            "document": results["documents"][0][i],
            "distance": results["distances"][0][i]
        })
    return retrieved

In [32]:
# ====================== PHASE 2: LangGraph + RAG Recruiter ======================

from langgraph.graph import StateGraph, END
from typing import TypedDict, List, Optional, Any

class RecruiterState(TypedDict):
    jd_raw_text: str
    top_n: int
    retrieve_k: int
    job_requirements: Optional[Any]
    retrieved: List[dict]
    all_candidates: List[Any]
    matches: List[Any]
    top_matches: List[Any]
    report: str
    error: Optional[str]


def node_analyze_jd(state: RecruiterState) -> RecruiterState:
    print("→ [LangGraph] Analyzing JD...")
    try:
        jd = parse_job_description(state["jd_raw_text"])
        state["job_requirements"] = jd
        print(f"✓ Job: {jd.job_title}")
    except Exception as e:
        state["error"] = f"JD analysis failed: {e}"
    return state


def node_retrieve(state: RecruiterState) -> RecruiterState:
    if state.get("error"):
        return state
    print("→ [LangGraph] Retrieving candidates from Vector DB...")
    try:
        retrieved = retrieve_candidates(state["jd_raw_text"], top_k=state.get("retrieve_k", 8))
        state["retrieved"] = retrieved
        state["all_candidates"] = load_all_candidates()
        print(f"✓ Retrieved {len(retrieved)} candidates")
        for r in retrieved:
            print(f"   - {r['name']} (distance={r['distance']:.3f})")
    except Exception as e:
        state["error"] = f"Retrieval failed: {e}"
    return state


def node_match(state: RecruiterState) -> RecruiterState:
    if state.get("error"):
        return state
    print("→ [LangGraph] LLM matching retrieved candidates...")
    try:
        jd = state["job_requirements"]
        name_map = {c.name.upper(): c for c in state["all_candidates"]}
        matches = []

        for r in state["retrieved"]:
            cand = name_map.get(r["name"].upper())
            if not cand:
                continue
            match = match_candidate_to_jd(cand, jd)
            matches.append(match)
            print(f"   {match.candidate_name}: {match.overall_score}")

        matches.sort(key=lambda x: x.overall_score, reverse=True)
        state["matches"] = matches
        state["top_matches"] = matches[: state.get("top_n", 5)]
        print(f"✓ Top {len(state['top_matches'])} selected")
    except Exception as e:
        state["error"] = f"Matching failed: {e}"
    return state


def node_questions(state: RecruiterState) -> RecruiterState:
    if state.get("error"):
        return state
    print("→ [LangGraph] Generating interview questions...")
    try:
        state["top_matches"] = enrich_top_candidates_with_questions(
            state["top_matches"],
            state["all_candidates"],
            state["job_requirements"]
        )
        print("✓ Questions generated")
    except Exception as e:
        state["error"] = f"Question generation failed: {e}"
    return state


def node_report(state: RecruiterState) -> RecruiterState:
    if state.get("error"):
        return state
    print("→ [LangGraph] Building recruiter report...")
    try:
        # Reuse your existing printer (prints to stdout)
        print_recruiter_report(state["job_requirements"], state["top_matches"])
        state["report"] = "Report generated successfully"
    except Exception as e:
        state["error"] = f"Report failed: {e}"
    return state


def create_recruiter_graph():
    workflow = StateGraph(RecruiterState)

    workflow.add_node("analyze_jd", node_analyze_jd)
    workflow.add_node("retrieve", node_retrieve)
    workflow.add_node("match", node_match)
    workflow.add_node("questions", node_questions)
    workflow.add_node("report", node_report)

    workflow.set_entry_point("analyze_jd")
    workflow.add_edge("analyze_jd", "retrieve")
    workflow.add_edge("retrieve", "match")
    workflow.add_edge("match", "questions")
    workflow.add_edge("questions", "report")
    workflow.add_edge("report", END)

    return workflow.compile()


recruiter_agent = create_recruiter_graph()
print("Recruiter LangGraph (RAG) compiled!")


def run_recruiter_pipeline_rag(jd_text: str, top_n: int = 5, retrieve_k: int = 8):
    """
    LangGraph entrypoint for Recruiter RAG pipeline.
    Keep this name so Gradio can call it the same way.
    """
    initial_state = {
        "jd_raw_text": jd_text,
        "top_n": top_n,
        "retrieve_k": retrieve_k,
        "job_requirements": None,
        "retrieved": [],
        "all_candidates": [],
        "matches": [],
        "top_matches": [],
        "report": "",
        "error": None,
    }

    final_state = recruiter_agent.invoke(initial_state)

    if final_state.get("error"):
        print("Pipeline Error:", final_state["error"])
        return None

    return {
        "job_requirements": final_state["job_requirements"],
        "retrieved": final_state["retrieved"],
        "top_candidates": final_state["top_matches"],
    }

print("run_recruiter_pipeline_rag (LangGraph) ready!")

Recruiter LangGraph (RAG) compiled!
run_recruiter_pipeline_rag (LangGraph) ready!


In [33]:
sample_jd = """
Job Title: Backend Engineer

Required Skills:
- Python, Go
- Microservices and distributed systems
- PostgreSQL, Redis, Kafka
- Docker and Kubernetes

Preferred:
- AWS
- Fintech experience

Experience: 2+ years
"""

result = run_recruiter_pipeline_rag(sample_jd, top_n=5, retrieve_k=8)

→ [LangGraph] Analyzing JD...
✓ Job: Backend Engineer
→ [LangGraph] Retrieving candidates from Vector DB...
✓ Retrieved 8 candidates
   - NOAH KIM (distance=0.265)
   - MOHAMMED AL-FARSI (distance=0.300)
   - ISABELLA ROSSI (distance=0.330)
   - MAYA PATEL (distance=0.345)
   - JORDAN LEE (distance=0.345)
   - CHEN WEI (distance=0.371)
   - SOPHIE MÜLLER (distance=0.396)
   - AARAV MEHTA (distance=0.412)
→ [LangGraph] LLM matching retrieved candidates...
   NOAH KIM: 58.0
   MOHAMMED AL-FARSI: 52.0
   ISABELLA ROSSI: 65.0
   MAYA PATEL: 68.0
   JORDAN LEE: 52.0
   CHEN WEI: 82.0
   SOPHIE MÜLLER: 55.0
   AARAV MEHTA: 82.0
✓ Top 5 selected
→ [LangGraph] Generating interview questions...
Generating interview questions for Top candidates...

[1] Generating questions for CHEN WEI
    ✓ 8 questions generated
[2] Generating questions for AARAV MEHTA
    ✓ 8 questions generated
[3] Generating questions for MAYA PATEL
    ✓ 8 questions generated
[4] Generating questions for ISABELLA ROSSI
    

In [34]:
!pip install -q gradio

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pdfplumber 0.11.10 requires Pillow>=12.2.0, but you have pillow 11.3.0 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.


In [35]:
import gradio as gr

# ====================== CANDIDATE PIPELINE ======================
def candidate_pipeline(resume_file, target_role):
    if resume_file is None:
        return "Please upload a resume PDF.", "", "", "", "", "", ""

    try:
        resume_path = resume_file if isinstance(resume_file, str) else resume_file.name
        result = run_careerforge(resume_path, target_role=target_role if target_role else None)

        if result.get("error"):
            return f"Error: {result['error']}", "", "", "", "", "", ""

        resume = result["structured_resume"]
        analysis = result["resume_analysis"]
        gen = result["generated_content"]

        score_text = f"""**Candidate:** {resume.name}
**Target Role:** {target_role or 'Not specified'}
**Resume Score:** {analysis.overall_score}/100
**Status:** {analysis.approval_status}"""

        strengths = "\n".join([f"✓ {s}" for s in analysis.strengths]) or "None"
        weaknesses = "\n".join([f"✗ {w}" for w in analysis.weaknesses]) or "None"

        missing_skills = "\n".join([f"→ {s}" for s in analysis.missing_skills_for_role]) or "None"
        certs = "\n".join([f"→ {c}" for c in analysis.recommended_certifications]) or "None"
        projects = "\n".join([f"→ {p}" for p in analysis.recommended_projects]) or "None"

        recommendations = f"""**Missing Skills:**
{missing_skills}

**Recommended Certifications:**
{certs}

**Recommended Projects:**
{projects}

**Feedback:**
{analysis.feedback_summary}"""

        improved_summary = gen.improved_summary or "Not generated"

        suggestions = ""
        if gen.resume_suggestions:
            for i, s in enumerate(gen.resume_suggestions, 1):
                suggestions += f"{i}. Original: {s.original_bullet}\n"
                suggestions += f"   Improved: {s.improved_bullet}\n"
                suggestions += f"   Reason: {s.reason}\n\n"
        else:
            suggestions = "No major suggestions."

        full_resume = gen.full_tailored_resume_text or "Not generated"
        questions = "\n".join(
            [f"{i}. {q}" for i, q in enumerate(gen.interview_questions, 1)]
        ) or "None"

        return (
            score_text,
            strengths,
            weaknesses,
            recommendations,
            improved_summary,
            suggestions,
            full_resume + "\n\n--- Interview Questions ---\n" + questions,
        )

    except Exception as e:
        return f"Pipeline error: {str(e)}", "", "", "", "", "", ""


# ====================== RECRUITER PIPELINE (RAG + LangGraph) ======================
def recruiter_pipeline(jd_text, top_n):
    if not jd_text or not jd_text.strip():
        return "Please paste a Job Description.", ""

    try:
        top_n = int(top_n) if top_n else 5

        # ✅ Uses RAG + LangGraph version
        result = run_recruiter_pipeline_rag(
            jd_text,
            top_n=top_n,
            retrieve_k=8
        )

        if not result:
            return "No results or pipeline failed.", ""

        jd = result["job_requirements"]
        top_matches = result["top_candidates"]
        retrieved = result.get("retrieved", [])

        header = f"""**Job Title:** {jd.job_title}
**Required Skills:** {', '.join(jd.required_skills[:8])}
**Experience Level:** {jd.experience_level}
**Vector Retrieved:** {len(retrieved)} candidates
**Final Shortlist:** {len(top_matches)}
"""

        report = ""
        for rank, match in enumerate(top_matches, 1):
            report += f"""
{'='*60}
#{rank}  {match.candidate_name}  |  Score: {match.overall_score}/100
{'='*60}

Strengths:
{chr(10).join(['  ✓ ' + s for s in match.strengths]) or '  None'}

Gaps:
{chr(10).join(['  ✗ ' + g for g in match.gaps]) or '  None'}

Summary:
{match.summary}

Interview Questions:
{chr(10).join([f'  {i}. {q}' for i, q in enumerate(match.interview_questions[:6], 1)])}
"""

        return header, report

    except Exception as e:
        return f"Recruiter pipeline error: {str(e)}", ""


# ====================== GRADIO APP ======================
with gr.Blocks(title="CareerForge AI") as demo:
    gr.Markdown("# CareerForge AI")
    gr.Markdown("Candidate Mode + Recruiter Mode (RAG + LangGraph + Vector DB)")

    with gr.Tabs():

        # ---------- CANDIDATE TAB ----------
        with gr.Tab("Candidate Mode"):
            gr.Markdown("### Upload a resume for analysis, improvement, and interview prep")

            with gr.Row():
                with gr.Column(scale=1):
                    c_resume = gr.File(label="Upload Resume (PDF)", file_types=[".pdf"])
                    c_role = gr.Textbox(
                        label="Target Role (optional)",
                        placeholder="e.g. Backend Engineer, AI Engineer"
                    )
                    c_btn = gr.Button("Analyze Resume", variant="primary")

                with gr.Column(scale=2):
                    c_score = gr.Markdown()
                    c_strengths = gr.Textbox(label="Strengths", lines=4)
                    c_weaknesses = gr.Textbox(label="Weaknesses", lines=4)

            c_recommendations = gr.Textbox(label="Career Recommendations", lines=8)
            c_summary = gr.Textbox(label="Improved Summary", lines=3)
            c_suggestions = gr.Textbox(label="Resume Suggestions", lines=8)
            c_full = gr.Textbox(label="Full Improved Resume + Interview Questions", lines=18)

            c_btn.click(
                fn=candidate_pipeline,
                inputs=[c_resume, c_role],
                outputs=[
                    c_score, c_strengths, c_weaknesses,
                    c_recommendations, c_summary, c_suggestions, c_full
                ]
            )

        # ---------- RECRUITER TAB ----------
        with gr.Tab("Recruiter Mode"):
            gr.Markdown("### Paste a JD → Vector search + LLM shortlist from database")

            with gr.Row():
                with gr.Column(scale=1):
                    r_jd = gr.Textbox(
                        label="Job Description",
                        lines=14,
                        placeholder="Paste full JD here..."
                    )
                    r_topn = gr.Number(label="Top N Candidates", value=5, precision=0)
                    r_btn = gr.Button("Find Matching Candidates", variant="primary")

                with gr.Column(scale=2):
                    r_header = gr.Markdown()
                    r_report = gr.Textbox(label="Shortlist Report", lines=28)

            r_btn.click(
                fn=recruiter_pipeline,
                inputs=[r_jd, r_topn],
                outputs=[r_header, r_report]
            )

demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://59964bf9305ee42fcd.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


         CareerForge AI - Phase 1 (Candidate Side)
→ Running Resume Parser...
✓ Resume parsed: Isparsh Chauhan
→ Running Resume Analyzer...
✓ Analyzed | Score: 78.0 | Status: Needs Improvement
→ Running Content Generator...
✓ Content generated!
  - Summary      : Yes
  - Suggestions  : 4
  - Questions    : 8
  - Full Resume  : Yes

Pipeline Completed Successfully!
→ [LangGraph] Analyzing JD...
✓ Job: Backend Engineer
→ [LangGraph] Retrieving candidates from Vector DB...
✓ Retrieved 8 candidates
   - MOHAMMED AL-FARSI (distance=0.333)
   - NOAH KIM (distance=0.334)
   - JORDAN LEE (distance=0.364)
   - MAYA PATEL (distance=0.370)
   - ISABELLA ROSSI (distance=0.389)
   - CHEN WEI (distance=0.399)
   - SOPHIE MÜLLER (distance=0.405)
   - ARJUN DESAI (distance=0.427)
→ [LangGraph] LLM matching retrieved candidates...
   MOHAMMED AL-FARSI: 48.0
   NOAH KIM: 52.0
   JORDAN LEE: 55.0
   MAYA PATEL: 70.0
   ISABELLA ROSSI: 55.0
   CHEN WEI: 78.0
   SOPHIE MÜLLER: 48.0
   ARJUN DESAI: 45.0
✓ T